Audd felismerés

In [ ]:
pip install audd

In [ ]:
import os
import json
import numpy as np
import urllib.request

In [ ]:
from audd import AudD
token = "test"
audd = AudD(token)

def recognize_song(file_path):
    result = audd.recognize(file_path)
    if result:
        return result.artist, result.title
    else:
        return "nem talalhato"


Essentia elemzés

In [ ]:
pip install essentia-tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.5/291.5 MB 10.8 MB/s  0:00:26


In [ ]:

from essentia.standard import MonoLoader, TensorflowPredictEffnetDiscogs, TensorflowPredict2D, TensorflowPredictMusiCNN, TempoCNN


BASE = "https://essentia.upf.edu/models"
MODELS = {
    "embedding":      "feature-extractors/discogs-effnet/discogs-effnet-bs64-1",
    "genre":          "classification-heads/genre_discogs400/genre_discogs400-discogs-effnet-1",
    "happy":          "classification-heads/mood_happy/mood_happy-discogs-effnet-1",
    "sad":            "classification-heads/mood_sad/mood_sad-discogs-effnet-1",
    "relaxed":        "classification-heads/mood_relaxed/mood_relaxed-discogs-effnet-1",
    "aggressive":     "classification-heads/mood_aggressive/mood_aggressive-discogs-effnet-1",
    "party":          "classification-heads/mood_party/mood_party-discogs-effnet-1",
    "danceability":   "classification-heads/danceability/danceability-discogs-effnet-1",
    "voice":          "classification-heads/voice_instrumental/voice_instrumental-discogs-effnet-1",
    "instruments":    "classification-heads/mtg_jamendo_instrument/mtg_jamendo_instrument-discogs-effnet-1",
    "timbre":         "classification-heads/timbre/timbre-discogs-effnet-1",
    "acoustic":       "classification-heads/mood_acoustic/mood_acoustic-discogs-effnet-1",
    "electronic":     "classification-heads/mood_electronic/mood_electronic-discogs-effnet-1",
    "deam":           "classification-heads/deam/deam-msd-musicnn-2",
    "musicnn":        "feature-extractors/musicnn/msd-musicnn-1",
    "tempocnn":       "tempo/tempocnn/deeptemp-k16-3",

}

os.makedirs("models", exist_ok=True)
for name, path in MODELS.items():
    for ext in (".pb", ".json"):
       url = f"{BASE}/{path}{ext}"
       dest = os.path.join("models", os.path.basename(url))
       if not os.path.exists(dest):
        urllib.request.urlretrieve(url, dest)

def model_files(name):
    fname = os.path.basename(MODELS[name])
    return f"models/{fname}.pb", f"models/{fname}.json"


def load_meta(name):
    x, meta_path = model_files(name)
    with open(meta_path) as f:
        return json.load(f)


def io_names(meta, purpose):
    schema = meta.get("schema", {})
    inputs = schema.get("inputs", [])
    outputs = schema.get("outputs", [])
    inp = inputs[0]["name"]
    out = next((o["name"] for o in outputs if o.get("output_purpose") == purpose))
    return inp, out


_emb_meta = load_meta("embedding")
x, _emb_out = io_names(_emb_meta, "embeddings")
embedder = TensorflowPredictEffnetDiscogs(
    graphFilename=model_files("embedding")[0],
    output=_emb_out or "PartitionedCall:1",
)


def get_embeddings(path):
    audio = MonoLoader(filename=path, sampleRate=16000, resampleQuality=4)()
    return embedder(audio)


def predict(name, embeddings):
    meta = load_meta(name)
    inp, out = io_names(meta, "predictions")
    kwargs = {"graphFilename": model_files(name)[0]}
    if inp:
        kwargs["input"] = inp
    if out:
        kwargs["output"] = out
    model = TensorflowPredict2D(**kwargs)
    scores = np.mean(model(embeddings), axis=0)
    return dict(zip(meta["classes"], scores.tolist()))


tempo_model = TempoCNN(graphFilename=model_files("tempocnn")[0])

def get_tempo(path):
    audio = MonoLoader(filename=path, sampleRate=11025, resampleQuality=4)()
    global_tempo, local_tempo, _ = tempo_model(audio)
    bpm = float(global_tempo)
    local = [round(float(t)) for t in local_tempo]
    return {
        "local_bpm": local,
        "global_bpm": round(bpm, 2),
    }

musicnn_embedder = TensorflowPredictMusiCNN(graphFilename=model_files("musicnn")[0],
                                            output="model/dense/BiasAdd")
av_model = TensorflowPredict2D(graphFilename=model_files("deam")[0], output="model/Identity")



def get_arousal_valence(path):
    audio = MonoLoader(filename=path, sampleRate=16000, resampleQuality=4)()
    emb = musicnn_embedder(audio)
    valence, arousal = np.mean(av_model(emb), axis=0)
    return {
        "arousal": round(float(arousal), 2),
        "valence": round(float(valence), 2),
        "energy": float(arousal),
        "positivity": float(valence),
    }

def analyze(path):
    emb = get_embeddings(path)
    voice = predict("voice", emb)
    return {
        #"genre": predict("genre", emb),
        "mood": {m: predict(m, emb)
                 for m in ("happy", "sad", "relaxed", "aggressive", "party")},
        "danceability": predict("danceability", emb),
        "voice": max(voice, key=voice.get),
        "instruments": dict(sorted(predict("instruments", emb).items(), key=lambda x: x[1], reverse=True)),
        "timbre": {c: round(s, 3) for c, s in predict("timbre", emb).items()},
        "tempo": get_tempo(path),
        "acoustic": predict("acoustic", emb),
        "electronic": predict("electronic", emb),
        "emotion": get_arousal_valence(path),
    }


Groq openai felismerés + ajánlás

In [ ]:
pip install -q openai

In [ ]:
from openai import OpenAI
from google.colab import userdata

def rnr(es_ans):
  client = OpenAI(base_url="https://api.groq.com/openai/v1",
                  api_key=userdata.get("music_analyzer"))


  question = "Egy zenenek a tulajdonsagait kivettunk Essentia-val. Kitalalod a zene mufajat és almufajat? Tegyel 5 tippet. Végülül ajanlj 3 masik, letezo zenet a megadott tulajdonsagok alapjan. Tomoren, csak a valaszokat add meg."
  data = json.dumps(es_ans, indent=2, ensure_ascii=False)
  response = client.chat.completions.create(
      model="openai/gpt-oss-120b",
      messages=[{"role": "user", "content": question},
                {"role": "user", "content": data}
              ]

  )
  return response.choices[0].message.content

In [ ]:


#audd rec
team_sample1_name = recognize_song("sample1.mp3")
team_sample2_name = recognize_song("sample2.mp3")
team_sample3_name = recognize_song("sample3.mp3")


#essentia elemzes
team_sample1_attributes = analyze("sample1.mp3")
team_sample2_attributes = analyze("sample2.mp3")
team_sample3_attributes = analyze("sample3.mp3")

#felismerés + ajánlás
team_sample1_rnr = rnr(team_sample1_attributes)
team_sample2_rnr = rnr(team_sample2_attributes)
team_sample3_rnr = rnr(team_sample3_attributes)



In [ ]:
#sample1
print("1. zene (on and on) -----------------------------------------------")
print("zenecím becslése: ", team_sample1_name) #audd
print("kinyert tulajdonságok: ", json.dumps(team_sample1_attributes, indent=2, ensure_ascii=False)) #essentia
print("Groq válasza: ", team_sample1_rnr) #groq


#sample2
print("\n\n2. zene (rollin) --------------------------------------------------")
print("zenecím becslése: ", team_sample2_name) #audd
print("kinyert tulajdonságok: ", json.dumps(team_sample2_attributes, indent=2, ensure_ascii=False)) #essentia
print("Groq válasza: ", team_sample2_rnr) #groq


#sample3
print("\n\n3. zene (citadella) -----------------------------------------------")
print("zenecím becslése: ", team_sample3_name) #audd
print("kinyert tulajdonságok: ", json.dumps(team_sample3_attributes, indent=2, ensure_ascii=False)) #essentia
print("Groq válasza: ", team_sample3_rnr) #groq

1. zene (on and on) -----------------------------------------------
zenecím becslése:  ('Cartoon', 'On & On')
kinyert tulajdonságok:  {
  "mood": {
    "happy": {
      "happy": 0.23567329347133636,
      "non_happy": 0.7643269300460815
    },
    "sad": {
      "non_sad": 0.7888800501823425,
      "sad": 0.21112015843391418
    },
    "relaxed": {
      "non_relaxed": 0.3767625689506531,
      "relaxed": 0.6232373714447021
    },
    "aggressive": {
      "aggressive": 0.05400330573320389,
      "not_aggressive": 0.9459966421127319
    },
    "party": {
      "non_party": 0.18004831671714783,
      "party": 0.8199517726898193
    }
  },
  "danceability": {
    "danceable": 0.9658067226409912,
    "not_danceable": 0.03419309854507446
  },
  "voice": "voice",
  "instruments": {
    "accordion": 0.004636505153030157,
    "acousticbassguitar": 0.0043447730131447315,
    "acousticguitar": 0.03656429797410965,
    "bass": 0.5621290802955627,
    "beat": 0.009777609258890152,
    "bell": 0.0